In [1]:
# Libraries
import json
import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

In [2]:
# Load the sales history (Nov 2021 - Aug 2026); in Google Colab, upload data/sales_history.csv when asked
import os
DATA_FILE = 'data/sales_history.csv'
if not os.path.exists(DATA_FILE):
    from google.colab import files
    DATA_FILE = next(iter(files.upload()))

sales = pd.read_csv(DATA_FILE, parse_dates=['ordered_date'])
print(f"{len(sales):,} sales, PHP {sales['total_amount'].sum():,.0f}, "
      f"{sales['ordered_date'].min():%Y-%m-%d} to {sales['ordered_date'].max():%Y-%m-%d}")

2,484 sales, PHP 2,111,386, 2021-11-01 to 2026-08-31


In [3]:
# Yearly, monthly and weekly tables: Total Sales (product performance) and No. of Orders (customer purchase pattern)
def period_table(level):
    d = sales['ordered_date']
    if level == 'yearly':
        key = d.dt.year
        periods = list(range(key.min(), key.max() + 1))
        labels = [str(p) for p in periods]
    elif level == 'monthly':
        key = d.dt.to_period('M')
        periods = list(pd.period_range(key.min(), key.max(), freq='M'))
        labels = [p.strftime('%b %Y') for p in periods]
    else:
        key = (d - pd.to_timedelta(d.dt.weekday, unit='D')).dt.normalize()
        periods = list(pd.date_range(key.min(), key.max(), freq='7D'))
        labels = [f'Week of {p:%b %d, %Y}' for p in periods]
    grouped = sales.groupby(key)
    table = pd.DataFrame({
        'period': labels,
        'total_sales': grouped['total_amount'].sum().reindex(periods, fill_value=0).values,
        'orders': grouped.size().reindex(periods, fill_value=0).values,
    })
    # X = time: the year itself for yearly; 1, 2, 3 ... for monthly (Nov 2021 = 1) and weekly (week of Nov 1, 2021 = 1)
    table.insert(1, 'X', periods if level == 'yearly' else range(1, len(periods) + 1))
    return table

tables = {level: period_table(level) for level in ['yearly', 'monthly', 'weekly']}
for level, table in tables.items():
    table.to_csv(f'data/{level}.csv', index=False)
    print(f"{level:8s}: {len(table):3d} periods, PHP {table['total_sales'].sum():,.0f}, {table['orders'].sum():,} orders")
tables['yearly']

yearly  :   6 periods, PHP 2,111,386, 2,484 orders
monthly :  58 periods, PHP 2,111,386, 2,484 orders
weekly  : 253 periods, PHP 2,111,386, 2,484 orders


,period,X,total_sales,orders
0,2021,2021,72791,86
1,2022,2022,449084,514
2,2023,2023,437498,514
3,2024,2024,433426,515
4,2025,2025,437365,514
5,2026,2026,281222,341


In [4]:
# Train the Simple Linear Regression (OLS): Y = a + bX
TARGETS = {'product_performance': 'total_sales', 'customer_purchase_pattern': 'orders'}

def train(table, target):
    fit = sm.OLS(table[target].astype(float), sm.add_constant(table['X'].astype(float))).fit()
    return {
        'constant': float(fit.params['const']),
        'slope': float(fit.params['X']),
        'n': int(fit.nobs),
        'r': float(np.sqrt(fit.rsquared)),
        'r2': float(fit.rsquared),
        'adj_r2': float(fit.rsquared_adj),
        'f': float(fit.fvalue),
        'p': float(fit.f_pvalue),
        'df': [int(fit.df_model), int(fit.df_resid)],
    }

models = {name: {level: train(tables[level], target) for level in tables} for name, target in TARGETS.items()}

In [5]:
# Results of the six models
rows = []
for name, by_level in models.items():
    for level, m in by_level.items():
        rows.append({
            'model': name.replace('_', ' '), 'level': level, 'n': m['n'],
            'constant (a)': round(m['constant'], 4), 'slope (b)': round(m['slope'], 4),
            'R': round(m['r'], 3), 'R2': round(m['r2'], 3), 'adj R2': round(m['adj_r2'], 3),
            'F': f"F({m['df'][0]},{m['df'][1]}) = {m['f']:.3f}", 'p-value': round(m['p'], 3),
            'significant': 'yes' if m['p'] < 0.05 else 'no',
        })
results = pd.DataFrame(rows)
results

,model,level,n,constant (a),slope (b),R,R2,adj R2,F,p-value,significant
0,product performance,yearly,6,-5.763155e+07,28655.0286,0.356,0.126,-0.092,"F(1,4) = 0.579",0.489,no
1,product performance,monthly,58,3.709691e+04,-23.5153,0.085,0.007,-0.011,"F(1,56) = 0.403",0.528,no
2,product performance,weekly,253,8.605658e+03,-2.0493,0.069,0.005,0.001,"F(1,251) = 1.187",0.277,no
3,customer purchase pattern,yearly,6,-7.335703e+04,36.4571,0.390,0.152,-0.060,"F(1,4) = 0.716",0.445,no
4,customer purchase pattern,monthly,58,4.280040e+01,0.0009,0.013,0.000,-0.018,"F(1,56) = 0.009",0.926,no
5,customer purchase pattern,weekly,253,9.930100e+00,-0.0009,0.098,0.010,0.006,"F(1,251) = 2.421",0.121,no


In [6]:
# Save the trained model (used by 02_forecast_and_accuracy.ipynb and copied into ../model.js)
trained = {
    'model': 'Simple Linear Regression (OLS), Y = a + bX',
    'trained_on': {
        'from': f"{sales['ordered_date'].min():%Y-%m-%d}",
        'to': f"{sales['ordered_date'].max():%Y-%m-%d}",
        'sales': int(len(sales)),
        'total_sales': float(sales['total_amount'].sum()),
    },
    'x': {'yearly': 'year (e.g. 2027)', 'monthly': 'month no. (Nov 2021 = 1)', 'weekly': 'week no. (week of Nov 1, 2021 = 1)'},
    'models': models,
}
with open('trained_model.json', 'w') as f:
    json.dump(trained, f, indent=2)
print('Saved trained_model.json')

Saved trained_model.json
